In [1]:
import pandas as pd

df = pd.concat([pd.read_csv("data/raw/fraudTrain.csv/fraudTrain.csv"),
                pd.read_csv("data/raw/fraudTest.csv/fraudTest.csv")], ignore_index=True)
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
df["ts"] = pd.to_datetime(df["trans_date_trans_time"])

print(df.shape)
print("Date range:", df["ts"].min(), "to", df["ts"].max())
print("Fraud rate:", round(df["is_fraud"].mean() * 100, 3), "%")
print("Customers (cards):", df["cc_num"].nunique(), "| Merchants:", df["merchant"].nunique())
df.columns.tolist()

(1852394, 23)
Date range: 2019-01-01 00:00:18 to 2020-12-31 23:59:34
Fraud rate: 0.521 %
Customers (cards): 999 | Merchants: 693


['trans_date_trans_time',
 'cc_num',
 'merchant',
 'category',
 'amt',
 'first',
 'last',
 'gender',
 'street',
 'city',
 'state',
 'zip',
 'lat',
 'long',
 'city_pop',
 'job',
 'dob',
 'trans_num',
 'unix_time',
 'merch_lat',
 'merch_long',
 'is_fraud',
 'ts']

In [2]:
print("Missing values per column:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicate transaction IDs:", df["trans_num"].duplicated().sum())
print("\nAmount summary:")
print(df["amt"].describe().round(2))

Missing values per column:
 Series([], dtype: int64)

Duplicate transaction IDs: 0

Amount summary:
count    1852394.00
mean          70.06
std          159.25
min            1.00
25%            9.64
50%           47.45
75%           83.10
max        28948.90
Name: amt, dtype: float64


In [3]:
df["hour"] = df["ts"].dt.hour
df["month"] = df["ts"].dt.to_period("M")

print("Fraud rate by category (%):")
print((df.groupby("category")["is_fraud"].mean() * 100).sort_values(ascending=False).round(2))

print("\nFraud rate by hour of day (%):")
print((df.groupby("hour")["is_fraud"].mean() * 100).round(2).to_string())

print("\nTransactions and fraud rate by month:")
m = df.groupby("month").agg(transactions=("is_fraud", "size"), fraud_rate_pct=("is_fraud", "mean"))
m["fraud_rate_pct"] = (m["fraud_rate_pct"] * 100).round(3)
m

Fraud rate by category (%):
category
shopping_net      1.59
misc_net          1.30
grocery_pos       1.26
shopping_pos      0.63
gas_transport     0.41
misc_pos          0.28
grocery_net       0.27
travel            0.27
personal_care     0.22
entertainment     0.22
kids_pets         0.19
food_dining       0.16
home              0.15
health_fitness    0.15
Name: is_fraud, dtype: float64

Fraud rate by hour of day (%):
hour
0     1.36
1     1.35
2     1.30
3     1.32
4     0.10
5     0.13
6     0.09
7     0.12
8     0.10
9     0.10
10    0.09
11    0.10
12    0.09
13    0.10
14    0.11
15    0.11
16    0.10
17    0.10
18    0.12
19    0.11
20    0.11
21    0.11
22    2.60
23    2.55

Transactions and fraud rate by month:


,transactions,fraud_rate_pct
month,,
2019-01,52525,0.963
2019-02,49866,1.037
2019-03,70939,0.696
2019-04,68078,0.552
2019-05,72532,0.563
2019-06,86064,0.411
2019-07,86596,0.382
2019-08,87359,0.437
2019-09,70652,0.592
